# Kalman Filter

The Kalman filter is a fundamental algorithm when analyzing and learning from time series data. If you're familiar with the forwards/backwards (or Baum-Welch) algorithm used for discrete state, discrete observation hidden Markov models, then you already know the basics. The Kalman filter is the forwards algorithm generalized to non-discrete states and observations. In this post, we'll review the Kalman filter in the linear Gaussian case.

<!-- TEASER_END -->

## Linear Dynamical Systems

The Kalman filter was developed as a tracking algorithm. The idea is to infer the unobserved location or state of an object given noisy, indirect measurements over time. We can abstract from the idea of tracking by instead studying *linear dynamical systems* (LDS). An LDS is a probabilistic model of two processes: dynamics and measurement. The dynamics model describes how the unobserved state of the system changes from one time to the next, and the measurement model tells us how our observations depend on the underlying system state. The Kalman filter combines the dynamics and measurement models with observed data to estimate the system state over time.

The dynamics model is simply a conditional probability distribution of a state given the previous state. Think of this as the probabilistic analog of a difference equation (or differential equation in continuous time). We index time using integers $k \in \mathbb{Z}$, and write the state at time $k$ as $x_k \in \mathbb{R}^n$. In addition to the state vectors, we assume that there are inputs $u_k \in \mathbb{R}^m$ at each step that influence the dynamics. The dynamics model is

$$
x_{k+1} \mid x_k, u_k, F, G, Q \sim
\mathcal{N} \big( F x_k + G u_k , Q \big).
$$

The measurement model is also a conditional probability distribution. At each time $k$ we may be able to measure the underlying state and receive observations $y_k \in \mathbb{R}^o$. The measurement model is

$$
y_k \mid x_k, H, R \sim
\mathcal{N} \big( H x_k, R \big).
$$

For the rest of this post, we'll assume that the system parameters $\mathcal{S} \triangleq (F, G, H, Q, R)$ are known and fixed. To cut down on clutter in the notation, we'll implicitly condition on the system parameters in all of our probabilistic and distributional expressions.

## Filtering

The filtering algorithm takes three inputs: an initial belief (or distribution) about the latent state, a sequence of measurements, and a sequence of inputs. Given these inputs, the filtering algorithm involves repeating two fundamental operations: `correct` and `predict`.

Here's some Python code that sketches how the filter works.

In [1]:
# This code is incomplete; it does not include system parameters.
# These functions are meant to highlight the logic behind the Kalman filter only.

def kalman_filter(m, C, Y, U):
    for y, u in zip(Y, U):
        m, C = kalman_step(m, C, y, u)
    return m, C

def kalman_step(m, C, y, u):
    m_c, C_c = correct(m, C, y)
    m_p, C_p = predict(m_c, C_c, u)
    return m_p, C_p

The output of the filter is the belief about the latent state at the time step *after* the final observation and input. To complete the algorithm, we need to implement the `correct` and `predict` functions, which depend on the parameters of the system ($\mathcal{S}$, defined above).

### Correct

The `correct` operation adjusts our belief about the state in light of measurements. If you're familiar with Bayesian linear regression, then `correct` is simply the posterior distribution over the model weights given data. To compute the correction, we need to define a few new quantities. We assume that the pre-correction belief about $x_k$ is a multivariate normal; let $m$ and $C$ denote the mean and covariance.

Since the measurement model is also multivariate normal and the mean is a linear function of $x_k$, we can infer two things: the joint distribution over state and measurements is multivariate normal, and the corrected distribution over the state is also multivariate normal. Our goal is to obtain an expression for the latter.

When we parameterize a joint multivariate normal using its [marginal distribution](https://pschulam.github.io/posts/parameterizing_joint_gaussians/), we get an intuitive expression for the conditional distribution of one component given the others. If we write the joint distribution over $x_k$ and $y_k$ using the marginal parameterization as

$$
\begin{bmatrix} x_k \\ y_k \end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix} m_x \\ m_y \end{bmatrix}
,
\begin{bmatrix}
K_{xx} & K_{xy} \\
K_{xy}^\top & K_{yy}
\end{bmatrix}
\right),
$$

then the conditional distribution over $x_k$ given $y_k$ is

$$
x_k \mid y_k \sim
\mathcal{N}
\big(
m_x + K_{xy} K_{yy}^{-1} (y_k - m_y),
K_{xx} - K_{xy} K_{yy}^{-1} K_{xy}^\top
\big).
$$

Why is this intuitive? First, the updated mean of the latent state is the uncorrected mean $m_x$ plus a correction term that is a linear function of the difference between the observed and expected value of $y_k$. The corrected covariance is the deflated uncorrected covariance, suggesting that we've learned something (we've reduced variance). Exactly how much we've learned depends on $K_{yy}$ and $K_{xy}$.

This shows us that one way to implement `correct` is to (1) convert the conditional parameterization of the joint distribution over $x_k$ and $y_k$ to a marginal parameterization, and (2) compute the conditional distribution of $x_k$ given $y_k$ using the marginal parameterization. To do step one, we have:

$$
m_x = m,
$$

$$
K_{xx} = C,
$$

$$
m_y = H m,
$$

$$
K_{yy} = H C H^\top + R,
$$

$$
K_{xy} = C H^\top.
$$

Now we can plug these expressions into the conditional distribution of $x_k$ given $y_k$ to see that

$$
\mathbb{E} \big[ x_k \mid y_k \big] =
m + C H^\top ( H C H^\top + R )^{-1} ( y_k - H m ),
$$

and

$$
\textsf{Cov} \big[ x_k \mid y_k \big] =
C - C H^\top ( H C H^\top + R)^{-1} H C.
$$

To clean this up a bit, we can define the Cholesky decomposition of $K_{yy}$

$$
H C H^\top + R = L L^\top.
$$

Then define $Z \triangleq L^{-1} H C$ and $X \triangleq L^{-\top} Z$, which implies

$$
\mathbb{E} \big[ x_k \mid y_k \big] = m + X^\top (y_k - H m),
$$

and

$$
\textsf{Cov} \big[ x_k \mid y_k \big] = C - Z^\top Z.
$$

Here is some Python code implementing these ideas.

In [2]:
from functools import partial

import numpy as np

from numpy.linalg import cholesky
from scipy.linalg import solve_triangular

solve_tril = partial(solve_triangular, lower=True)
solve_triu = partial(solve_triangular, lower=False)

def correct(H, R, m, C, y):
    L = cholesky(H @ C @ H.T + R)
    Z = solve_tril(L, H @ C)
    X = solve_triu(L.T, Z)
    m_c = m + X.T @ (y - H @ m)
    C_c = C - Z.T @ Z
    return m_c, C_c

### Predict

To wrap up the implementation, we need to describe the `predict` function. This is easier than the `correct` function, so we're almost done. Let $m$ and $C$ denote the mean and covariance of the belief about $x_k$. Given the dynamics model in $\mathcal{S}$, the belief about $x_{k+1}$ is

$$
x_{k + 1} \sim \mathcal{N} \big( F m + G u_k, F C F^\top + Q \big).
$$

Here is a function implementing this in Python.

In [3]:
def predict(F, G, Q, m, C, u):
    m_p = F @ m + G @ u
    C_p = F @ C @ F.T + Q
    return m_p, C_p

## Conclusion

This post described filtering, introduced linear dynamical systems (LDS), and implemented the Kalman filter for Gaussian LDS models. In future posts, we'll describe what we can do once we've got the Kalman filter in our toolbox.

Here's Python code implementing our complete solution.

In [4]:
from functools import partial

import numpy as np

from numpy.linalg import cholesky
from scipy.linalg import solve_triangular

solve_tril = partial(solve_triangular, lower=True)
solve_triu = partial(solve_triangular, lower=False)

'''
Assume that the system variable has the following structure.

system = {
    'dyn': (F, G, Q),
    'obs': (H, R)
}
'''

def kalman_filter(system, m, C, Y, U):
    for y, u in zip(Y, U):
        m, C = kalman_step(system, m, C, y, u)
    return m, C

def kalman_step(system, m, C, y, u):
    m_c, C_c = correct(*system['obs'], m, C, y)
    m_p, C_p = predict(*system['dyn'], m_c, C_c, u)
    return m_p, C_p

def correct(H, R, m, C, y):
    L = cholesky(H @ C @ H.T + R)
    Z = solve_tril(L, H @ C)
    X = solve_triu(L.T, Z)
    m_c = m + X.T @ (y - H @ m)
    C_c = C - Z.T @ Z
    return m_c, C_c

def predict(F, G, Q, m, C, u):
    m_p = F @ m + G @ u
    C_p = F @ C @ F.T + Q
    return m_p, C_p